In [3]:
import cv2 as cv
import numpy as np
import mediapipe as mp
from collections import deque, Counter
from tensorflow.keras.models import load_model

# ------------------- CONFIG -------------------
MODEL_PATH = "my_model.keras"  # your trained CNN model
CONF_THRESHOLD = 0.5           # minimum confidence to accept prediction
SMOOTH_WINDOW = 10             # number of frames for smoothing
CLASSES = ['A','B','C','D','E','F','G','H','I','K','L','M',
           'N','O','P','Q','R','S','T','U','V','W','X','Y',
           'del','nothing','space']  # update based on your labels

# ------------------- LOAD MODEL -------------------
model = load_model(MODEL_PATH)
model_height, model_width, model_channels = model.input_shape[1:4]

# ------------------- MEDIAPIPE INIT -------------------
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.6)

# Colors
landmark_color = (0, 0, 255)       # Red (filled)
connection_color = (255, 255, 255) # White

# ------------------- SMOOTHING HISTORY -------------------
pred_history = deque(maxlen=SMOOTH_WINDOW)

# ------------------- CAMERA LOOP -------------------
cap = cv.VideoCapture(0)
while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv.flip(frame, 1)
    h, w, _ = frame.shape

    # ---- Create gray blurred background ----
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    gray = cv.cvtColor(gray, cv.COLOR_GRAY2BGR)
    blurred_bg = cv.GaussianBlur(gray, (35, 35), 0)

    # ---- Process hands ----
    rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    results = hands.process(rgb)

    pred_label = "nothing"
    confidence = 0

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get hand bounding box
            x_coords = [lm.x for lm in hand_landmarks.landmark]
            y_coords = [lm.y for lm in hand_landmarks.landmark]
            x_min = int(min(x_coords) * w) - 40
            y_min = int(min(y_coords) * h) - 40
            x_max = int(max(x_coords) * w) + 40
            y_max = int(max(y_coords) * h) + 40
            x_min, y_min = max(0, x_min), max(0, y_min)
            x_max, y_max = min(w, x_max), min(h, y_max)

            # ---- Preprocess for model ----
            hand_img = frame[y_min:y_max, x_min:x_max]
            if hand_img.size == 0:
                continue

            hand_img = cv.resize(hand_img, (model_width, model_height))
            if model_channels == 1:
                hand_img = cv.cvtColor(hand_img, cv.COLOR_BGR2GRAY)
                hand_img = np.expand_dims(hand_img, axis=-1)
            else:
                hand_img = cv.cvtColor(hand_img, cv.COLOR_BGR2RGB)

            hand_img = hand_img.astype("float32") / 255.0
            hand_img = np.expand_dims(hand_img, axis=0)

            preds = model.predict(hand_img, verbose=0)[0]
            confidence = np.max(preds)
            label_index = np.argmax(preds)

            if confidence < CONF_THRESHOLD:
                pred_label = "nothing"
            else:
                pred_label = CLASSES[label_index]

            pred_history.append(pred_label)

            # ---- Draw hand on gray background ----
            blurred_bg[y_min:y_max, x_min:x_max] = frame[y_min:y_max, x_min:x_max]

            # Draw landmarks (red filled + white lines)
            for connection in mp_hands.HAND_CONNECTIONS:
                start_idx, end_idx = connection
                start = hand_landmarks.landmark[start_idx]
                end = hand_landmarks.landmark[end_idx]
                x1, y1 = int(start.x * w), int(start.y * h)
                x2, y2 = int(end.x * w), int(end.y * h)
                cv.line(blurred_bg, (x1, y1), (x2, y2), connection_color, 2)
            for lm in hand_landmarks.landmark:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv.circle(blurred_bg, (cx, cy), 5, landmark_color, -1)

            # Draw prediction box
            cv.rectangle(blurred_bg, (x_min, y_min - 40), (x_max, y_min), (0, 0, 0), -1)
            cv.putText(blurred_bg, f"{pred_label} ({confidence*100:.1f}%)",
                       (x_min + 5, y_min - 10), cv.FONT_HERSHEY_DUPLEX, 1, (255, 255, 255), 2)

    # ---- Smooth prediction ----
    if len(pred_history) > 0:
        stable_label = Counter(pred_history).most_common(1)[0][0]
    else:
        stable_label = "nothing"

    # ---- Display final ----
    cv.putText(blurred_bg, f"Stable: {stable_label}",
               (10, 40), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv.imshow("ASL Detection (Gray Background)", blurred_bg)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()
